# 01: 環境構築

Google Colab の T4 GPU ランタイムで実行してください。

Runtime > Change runtime type > T4 GPU

In [ ]:
import subprocess
import torch

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("GPU not available. Runtime > Change runtime type > T4 GPU を選択してください。")

In [ ]:
!pip install pycolmap --quiet
!python -c "import pycolmap; print(f'pycolmap: {pycolmap.__version__}')"

In [ ]:
import os
if not os.path.isdir("/content/gaussian-splatting"):
    !git clone --depth 1 https://github.com/graphdeco-inria/gaussian-splatting \n        --recursive /content/gaussian-splatting
    print("Cloned.")
else:
    print("Already exists, skipping.")

In [ ]:
!pip install plyfile tqdm Pillow matplotlib --quiet
!cd /content/gaussian-splatting && grep -v "^torch" requirements.txt | grep -v "^#" | grep -v "^$" | xargs pip install --quiet
print("Done")

In [ ]:
!pip install cccl --quiet
!pip install /content/gaussian-splatting/submodules/diff-gaussian-rasterization --quiet
!pip install /content/gaussian-splatting/submodules/simple-knn --quiet
print("=== Build complete ===")

In [ ]:
import sys
import importlib

sys.path.insert(0, "/content/gaussian-splatting")

results = {}
for name in ["diff_gaussian_rasterization", "simple_knn", "pycolmap"]:
    try:
        importlib.import_module(name)
        results[name] = "OK"
    except ImportError as e:
        results[name] = f"FAILED: {e}"

for k, v in results.items():
    print(f"{k}: {v}")

if all(v == "OK" for v in results.values()):
    print("\n全モジュールのインストール確認完了。02_colmap_sfm.ipynb に進んでください。")
else:
    print("\nエラーあり。docs/troubleshooting.md を参照してください。")